# seattle waterfront tsunami debris tracking

debris tracking on NOAA bathymetry of elliott bay: 3m surge from the west pushes containers seeded at the waterfront

run `make all`, then `python combine_fgout_nc.py` to generate reqs for this nb

In [ ]:
%matplotlib inline
from pylab import *
from importlib import reload
from clawpack.visclaw import animation_tools
from IPython.display import HTML
from clawpack.geoclaw import fgout_tools
from scipy.interpolate import RegularGridInterpolator

In [ ]:
import sys
sys.path.insert(0, '../../src')
from geoclaw_debris import debris_tracking
reload(debris_tracking)

In [ ]:
# load fgout data
x, y, t, qoi_arrays = fgout_tools.read_netcdf_arrays('elliott_bay_fgout.nc', qois=['h', 'u', 'v', 'B'], verbose=True)
print(f'dx={x[1]-x[0]:.6f} deg, dy={y[1]-y[0]:.6f} deg, {len(t)} frames')
X, Y = meshgrid(x, y, indexing='ij')
depth = qoi_arrays['h'].transpose((1, 2, 0))
u_ms = qoi_arrays['u'].transpose((1, 2, 0))
v_ms = qoi_arrays['v'].transpose((1, 2, 0))
B_xy = qoi_arrays['B'][0]  # static topo, shape (nx, ny) — no .T needed

## interpolating functions

note: velocities are m/s but positions are degrees, so need to convert velocities to deg/s

In [ ]:
lat_mid = 0.5 * (y[0] + y[-1])
m_per_deg_lat = 111000. # 111km is about one degree of latitude
m_per_deg_lon = 111000. * cos(radians(lat_mid)) # shrinks with latitude
u_degs = u_ms / m_per_deg_lon
v_degs = v_ms / m_per_deg_lat

#make all the interpolator functions
def make_fcn(qoi):
    interp = RegularGridInterpolator((x, y, t), qoi, method='linear', fill_value=nan, bounds_error=False)
    return lambda x,y,t: interp((x,y,t))

depth_fcn = make_fcn(depth)
u_fcn = make_fcn(u_degs)
v_fcn = make_fcn(v_degs)

def depth_positive_fcn(x, y, t):
    h = depth_fcn(x, y, t)
    return ma.masked_where(h < 0.001, h)

COLORS = ['red', 'blue', 'green', 'orange', 'purple', 'cyan']

## bathymetry

In [ ]:
aspect = 1/cos(radians(lat_mid)) # corrects for longitude shrinking at this latitude so that the plot doesnt look weird
fig, ax = subplots(figsize=(10, 7))
ax.pcolormesh(X, Y, B_xy, cmap='terrain', vmin=-30, vmax=50)
ax.contour(X, Y, B_xy, levels=[0], colors='black', linewidths=1)
ax.set_title('bathymetry')
ax.set_aspect(aspect)
tight_layout()

## seed debris

shipping containers (~12m x 2.4m) at waterfront locations, passive advection only for now.

In [ ]:
reload(debris_tracking)
# first convert the dimensions to degrees
avg_deg_per_m = 0.5 * (1/m_per_deg_lon + 1/m_per_deg_lat)
L_long = 12.0 * avg_deg_per_m
L_short = 2.4 * avg_deg_per_m

# shallow water near shore (-1 to -5 m)
pier_locations = [
    (-122.3473, 47.590, 0.0),
    (-122.3463, 47.590, pi/6),
    (-122.3401, 47.605, 0.0),
    (-122.3463, 47.610, pi/4),
    (-122.3631, 47.620, 0.0),
    (-122.3559, 47.615, -pi/6),
]

debris_list = []
z0_list = []
# create the debris objects
for i, (lon, lat, theta) in enumerate(pier_locations):
    db = debris_tracking.DebrisObject()
    db.L = [L_short, L_long, L_short]
    db.phi = [pi/2, pi/2, pi/2]
    db.z = (lon, lat, theta)
    db.advect = True
    db.rho = 0.
    debris_list.append(db)
    z0_list.append(db.z)
    print(f'#{i}: ({lon:.4f}, {lat:.3f}), h0={depth_fcn(lon, lat, 0):.2f}m')

In [ ]:
# plot bathymetry with debris starting positions
fig, ax = subplots(figsize=(12, 8))
ax.pcolormesh(X, Y, B_xy, cmap='terrain', vmin=-30, vmax=50)
ax.contour(X, Y, B_xy, levels=[0], colors='black', linewidths=1)

for i, db in enumerate(debris_list):
    xc, yc = db.get_corners(db.z, close_poly=True)
    ax.plot(xc, yc, color='red', linewidth=2)
    ax.annotate(f'{i}', db.z[:2], fontsize=10, fontweight='bold',color='red', ha='center', va='bottom')

ax.set_aspect(aspect)
ax.set_xlim(-122.38, -122.33)
ax.set_ylim(47.585, 47.625)
tight_layout()

## compute debris paths

In [ ]:
domain = [x.min(), x.max(), y.min(), y.max()]
t0 = 60.  # start when the surge front arrives to waterfront
dt = 5.0; nsteps = 100  # t=60 to t=560 in total

debris_path_list = debris_tracking.make_debris_path_list(debris_list, z0_list, [], domain, t0, dt, nsteps, depth_fcn, u_fcn, v_fcn, verbose=False)
print('done')

## trajectories

In [ ]:
#plot the trajectories of the objects over the 500 secs
fig, ax = subplots(figsize=(12, 8))
ax.pcolormesh(X, Y, B_xy, cmap='terrain', vmin=-30, vmax=50, alpha=0.5)
ax.contour(X, Y, B_xy, levels=[0], colors='black', linewidths=1)

for i, dp in enumerate(debris_path_list):
    x_c = dp.x_path.mean(axis=1)
    y_c = dp.y_path.mean(axis=1)
    ax.plot(x_c, y_c, color=COLORS[i], linewidth=1.5, label=f'#{i}')
    ax.plot(x_c[0], y_c[0], 'o', color=COLORS[i], markersize=8)
    ax.plot(x_c[-1], y_c[-1], 's', color=COLORS[i], markersize=8)

ax.legend(fontsize=9, loc='upper left')
ax.set_aspect(aspect)
ax.set_xlim(-122.38, -122.33)
ax.set_ylim(47.585, 47.625)
tight_layout()

In [ ]:
#plot the speeds and water depths of each object over time
fig, axes = subplots(2, 1, figsize=(12, 7), sharex=True)
for i, dp in enumerate(debris_path_list):
    x_c = dp.x_path.mean(axis=1)
    y_c = dp.y_path.mean(axis=1)
    h_along = array([depth_fcn(xc, yc, tn) for xc, yc, tn in zip(x_c, y_c, dp.times)])
    u_deg = dp.u_path.mean(axis=1)
    v_deg = dp.v_path.mean(axis=1)
    speed = sqrt((u_deg * m_per_deg_lon)**2 + (v_deg * m_per_deg_lat)**2)
    axes[0].plot(dp.times, h_along, color=COLORS[i], label=f'#{i}')
    axes[1].plot(dp.times, speed, color=COLORS[i], label=f'#{i}')

axes[0].set_ylabel('depth (m)')
axes[0].legend(fontsize=8, ncol=3)
axes[0].grid(True, alpha=0.3)
axes[1].set_ylabel('speed (m/s)')
axes[1].set_xlabel('time (s)')
axes[1].legend(fontsize=8, ncol=3)
axes[1].grid(True, alpha=0.3)
tight_layout()

## animation

In [ ]:
from clawpack.visclaw import colormaps
cmap_water = colormaps.make_colormap({0:[0,1,1], 1:[0,0,1]})

figs = []
times = debris_path_list[0].times
for n in range(0, len(times), 4):
    t_n = times[n]
    fig, ax = subplots(figsize=(10, 7))
    ax.pcolormesh(X, Y, depth_positive_fcn(X, Y, t_n), cmap=cmap_water, vmin=0, vmax=6)
    ax.contour(X, Y, B_xy, levels=[0], colors='black', linewidths=0.5, alpha=0.5)
    for i, dp in enumerate(debris_path_list):
        xc, yc = debris_list[i].get_corners(dp.z_path[n], close_poly=True)
        ax.plot(xc, yc, color=COLORS[i], linewidth=2)
    ax.set_xlim(-122.38, -122.33)
    ax.set_ylim(47.585, 47.625)
    ax.set_aspect(aspect)
    ax.set_title(f't = {t_n:.0f} s')
    figs.append(fig)
    close(fig)

anim = animation_tools.animate_images(animation_tools.make_images(figs), figsize=(10, 7))
HTML(anim.to_jshtml())